# SpeechNet On-Device Training Tutorial

This tutorial walks through the complete pipeline for deploying **SpeechNet** (a lightweight CNN for EMG-based silent speech recognition) on the **Siracusa RISC-V MCU** using **Deeploy**.

You will learn:
1. How to define a Deeploy-friendly PyTorch model
2. How to export inference and training ONNX graphs using Onnx4Deeploy
3. How to run untiled and tiled Deeploy deployment on Siracusa (GVSoC)
4. Key design decisions and pitfalls

**Prerequisites**: Familiarity with PyTorch, ONNX, and basic knowledge of RISC-V MCU architectures.

**Reference**: Spacone et al., "SilentWear: an Ultra-Low Power Wearable System for EMG-based Silent Speech Recognition", arXiv: 2603.02847.

## 1. Model Architecture

SpeechNet is a 5-block CNN processing 14-channel EMG signals:

| Block | Conv kernel | In→Out channels | Output shape |
|-------|------------|-----------------|-------------|
| 0 | (1, 4) | 1 → 8 | (8, 14, 87) after AvgPool(1,8) |
| 1 | (1, 16) | 8 → 16 | (16, 14, 22) after AvgPool(1,4) |
| 2 | (1, 8) | 16 → 16 | (16, 14, 5) after AvgPool(1,4) |
| 3 | (7, 1) | 16 → 32 | (32, 8, 5) after AvgPool(1,1) |
| 4 | (7, 1) | 32 → 32 | (32, 2, 5) after AvgPool(1,1) |

Followed by GlobalAvgPool → Reshape → Linear(32, 9).

Total: ~15K parameters, 9 output classes (8 speech commands + rest).

## 2. Defining a Deeploy-Friendly PyTorch Model

When designing a model for Deeploy deployment, follow these rules:

### Rule 1: No dynamic ONNX ops
Avoid `torch.flatten()`, `x.size()`, `x.shape[N]` in the forward pass. These generate dynamic `Shape`/`Gather`/`Flatten` ops in ONNX that Deeploy cannot handle.

**Bad:**
```python
x = torch.flatten(x, 1)  # generates Flatten + Shape in backward
```

**Good:**
```python
x = x.reshape(1, self._fc_in)  # static reshape, batch=1 for deployment
```

### Rule 2: Use AvgPool instead of MaxPool
MaxPool gradient requires index storage. AvgPool gradient is a simple scatter-divide.

### Rule 3: No Dropout
Dropout is a no-op at inference and unnecessary for on-device fine-tuning.

In [ ]:
import torch
import torch.nn as nn
from typing import Any, Dict, List, Optional


class SpeechNetDeploy(nn.Module):
    """Deployment-ready SpeechNet for Deeploy on PULP MCUs."""

    def __init__(
        self,
        num_channels: int = 14,
        time_steps: int = 700,
        num_classes: int = 9,
        blocks_config: Optional[List[Dict[str, Any]]] = None,
    ):
        super().__init__()
        if blocks_config is None:
            blocks_config = [
                dict(out_channels=8, kernel=(1, 4), pool=(1, 8)),
                dict(out_channels=16, kernel=(1, 16), pool=(1, 4)),
                dict(out_channels=16, kernel=(1, 8), pool=(1, 4)),
                dict(out_channels=32, kernel=(7, 1), pool=(1, 1)),
                dict(out_channels=32, kernel=(7, 1), pool=(1, 1)),
            ]

        self.blocks = nn.ModuleList()
        in_ch = 1
        for cfg in blocks_config:
            out_ch = cfg["out_channels"]
            k_c, k_t = cfg["kernel"]
            pool_c, pool_t = cfg.get("pool", (1, 1))
            layers = [
                nn.Conv2d(in_ch, out_ch, kernel_size=(k_c, k_t),
                          padding=(0, k_t // 2), bias=True),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=False),  # inplace=False for clean ONNX
                nn.AvgPool2d(kernel_size=(pool_c, pool_t),
                             stride=(pool_c, pool_t)),
            ]
            self.blocks.append(nn.Sequential(*layers))
            in_ch = out_ch

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self._fc_in = in_ch  # stored as Python int for static reshape
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            x = block(x)
        x = self.global_pool(x)
        # Static reshape: avoids dynamic Shape/Flatten ops in ONNX
        x = x.reshape(1, self._fc_in)
        x = self.fc(x)
        return x


model = SpeechNetDeploy()
x = torch.randn(1, 1, 14, 700)
y = model(x)
print(f"Input: {x.shape} → Output: {y.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Exporting ONNX with Onnx4Deeploy

Onnx4Deeploy provides a unified CLI for exporting models to ONNX format compatible with Deeploy.

### 3.1 Inference Export

```bash
cd /path/to/Onnx4Deeploy
python Onnx4Deeploy.py -model SpeechNet -mode infer
```

This produces:
- `onnx/model/speechnet_infer/network.onnx` — inference graph (BN folded into Conv)
- `onnx/model/speechnet_infer/inputs.npz` — test input
- `onnx/model/speechnet_infer/outputs.npz` — reference output

### 3.2 Training Export

```bash
python Onnx4Deeploy.py -model SpeechNet -mode train
```

This produces:
- `onnx/model/speechnet_train/network.onnx` — training graph (forward + backward + gradient accumulation)
- `onnx/model/speechnet_train/inputs.npz` — multi-batch training data
- `onnx/model/speechnet_train/outputs.npz` — reference updated weights + losses
- `onnx/model/speechnet_optimizer/network.onnx` — SGD optimizer graph

In [ ]:
# Verify the training ONNX graph structure
import onnx
from collections import Counter

m = onnx.load("onnx/model/speechnet_train/network.onnx")
c = Counter(n.op_type for n in m.graph.node)
print(f"Total nodes: {len(m.graph.node)}")
print(f"Forward ops: Conv={c['Conv']}, BN={c['BatchNormInternal']}, Relu={c['Relu']}, AvgPool={c['AveragePool']}")
print(f"Backward ops: ConvGrad={c['ConvGrad']}, BNGrad={c['BatchNormalizationGrad']}, ReluGrad={c['ReluGrad']}")
print(f"Training ops: InPlaceAccumulatorV2={c['InPlaceAccumulatorV2']}, SoftmaxCELoss={c['SoftmaxCrossEntropyLoss']}")

# Check for dynamic ops (should be 0)
dynamic_ops = ['Shape', 'Flatten', 'Expand', 'Gather']
bad = {op: c[op] for op in dynamic_ops if c.get(op, 0) > 0}
assert not bad, f"Dynamic ops found: {bad}"
print("\n✅ Clean graph — no dynamic ops")

### 3.3 Training Strategies

You can control which layers are trainable:

```bash
# Full training (all layers)
python Onnx4Deeploy.py -model SpeechNet -mode train

# Last-layer only (transfer learning)
python Onnx4Deeploy.py -model SpeechNet -mode train --training-strategy last_layer
```

The training strategy controls the backward graph size:

| Strategy | Trainable params | Backward ops | Use case |
|----------|-----------------|-------------|----------|
| `full` | 22 | ConvGrad×5, BNGrad×5, ReluGrad×5, AvgPoolGrad×5 | Full fine-tuning |
| `last_layer` | 2 (fc only) | Gemm backward only | Quick adaptation |
| `custom` | User-defined | Depends on selection | Selective fine-tuning |

## 4. Deploying with Deeploy on Siracusa

### 4.1 Environment Setup

```bash
# Activate the TrainDeeploy environment
source /path/to/TrainDeeploy/activate_traindeeploy.sh
cd TrainDeeploy/DeeployTest
```

### 4.2 Untiled Deployment (Smoke Test)

Run the untiled version first to verify numerical correctness:

```bash
python deeployTrainingRunner_siracusa.py \
    -t /path/to/Onnx4Deeploy/onnx/model/speechnet_train
```

Expected output:
```
=== Siracusa Training Harness (Phase 2 — with OptimizerNetwork) ===
N_TRAIN_STEPS=4  N_ACCUM_STEPS=1  DATA_INPUTS=2
Initializing TrainingNetwork...
Initializing OptimizerNetwork...
Starting training (4 optimizer steps x 1 accum steps)...
update 1/4  accum 1/1  (mini-batch 0)
...
[loss 0] computed=2.267950  ref=2.267950  diff=0.000000  TOL=0.001000
[loss 1] computed=2.498553  ref=2.498553  diff=0.000000  TOL=0.001000
[loss 2] computed=2.083153  ref=2.083153  diff=0.000000  TOL=0.001000
[loss 3] computed=1.905963  ref=1.905963  diff=0.000000  TOL=0.001000
Errors: 0 out of 4
BENCH train_cycles=285250543 opt_cycles=429083 weight_sram=61956

✓ Test speechnet_train PASSED - No errors found
```

### 4.3 Tiled Deployment

For real MCU deployment, use tiling to fit within L1 memory:

```bash
python deeployTrainingRunner_tiled_siracusa.py \
    -t /path/to/Onnx4Deeploy/onnx/model/speechnet_train \
    --l1 128000 --l2 2000000
```

The tiler automatically splits large activations into tiles that fit in L1 (128 KB).

## 5. Understanding the Tiling Pipeline

Deeploy's tiling pipeline works as follows:

```
ONNX graph
    ↓
FrontEnd: graph lowering, node renaming, constant folding
    ↓  
Parse: match each node to a NodeMapper (Parser + Bindings)
    ↓
Broadcast: compute/update tensor shapes
    ↓
TypeCheck: select the best NodeBinding (Template + TypeChecker)
    ↓
Bind: hoist transient buffers (e.g., im2col), set up execution blocks
    ↓
Tile: OR-Tools solver finds tile dimensions under L1/L2 constraints
    ↓
CodeGen: render C code with per-tile DMA + kernel calls
    ↓
Build: compile with LLVM for RISC-V
    ↓
Simulate: run on GVSoC cycle-accurate simulator
```

### Key concepts:

- **TileConstraint**: Defines how each op can be tiled (which dims are free, which are pinned)
- **Transient buffers**: Scratch memory needed by kernels (e.g., im2col buffer for Conv)
- **Memory hierarchy**: L1 (128 KB SRAM, fast) → L2 (2 MB SRAM) → L3 (HyperFlash, slow)

## 6. Common Pitfalls and Solutions

### Pitfall 1: `torch.flatten` generates dynamic Shape ops
**Symptom**: Training graph has `Shape` + `Reshape` nodes from Flatten backward.
**Fix**: Use `x.reshape(1, C)` with static dimensions.

### Pitfall 2: ConvGradX Im2Col buffer exceeds L1
**Symptom**: Tiled training hangs — GVSoC runs but no output.
**Cause**: The Im2Col ConvGradX kernel gets `ctxtBufferSize` from full-op dimensions (e.g., 1.2 MB) but the actual L1 allocation is only ~120 KB. The kernel's `co_block` auto-tuning overestimates → L1 overflow.
**Fix**: Use the naive ConvGradX kernel (`referenceConvGradX2DTemplate`) which doesn't require im2col. Change in `Bindings.py`.

### Pitfall 3: ConvLayer.computeShapes corrupts bias shape
**Symptom**: `TypeError: 'int' object is not iterable` during graph export.
**Cause**: `inputShapes[2] = inputShapes[1][0]` sets bias shape to a scalar int instead of tuple.
**Fix**: `inputShapes[2] = (inputShapes[1][0],)` in `Layers.py`.

### Pitfall 4: Multiple GVSoC simulations sharing workdir
**Symptom**: `exitcode: -9` (SIGKILL) — simulations kill each other.
**Fix**: Use `PYTEST_XDIST_WORKER=<unique_id>` to isolate build directories.

### Pitfall 5: GVSoC stdout is fully buffered
**Symptom**: Simulation runs but no printf output visible.
**Fix**: Use `--trace=cluster/pe0/insn` to force output, or use `ring_tee.py` for bounded trace capture with heartbeat monitoring.

## 7. Debugging with GVSoC Traces

When a simulation hangs or produces wrong results, use GVSoC's built-in tracing:

### Trace FC (fabric controller) instructions
```bash
gvsoc --target=siracusa --binary=<bin> --work-dir=<dir> \
    --trace=fc/insn image flash run 2>trace_fc.txt
```
Shows every instruction the FC executes. Useful for finding where FC is stuck (e.g., `pi_task_wait_on` = waiting for cluster, `memcpy` = initializing data).

### Trace cluster PE instructions
```bash
gvsoc --target=siracusa --binary=<bin> --work-dir=<dir> \
    --trace=cluster/pe0/insn image flash run 2>trace_pe0.txt
```
Shows PE0's instructions. Look for the function name in the trace to identify which kernel is running:
```
125461135406: 9037685: [/chip/cluster/pe0/insn] PULP_Conv2d_Im2Col_fp32_fp32_f:0 M 1c031d58 flw ...
```

### Trace memory accesses (LSU)
```bash
--trace=cluster/pe0/lsu
```
Catches invalid memory accesses:
```
Invalid access (pc: 0x1c01c94c, offset: 0x3c9cf7a9, size: 0x3, is_write: 0)
```
This means a kernel tried to read address `0x3c9cf7a9` which is outside L1/L2 — indicates a buffer overflow or wrong DMA offset.

### Useful trace targets

| Trace flag | What it shows |
|-----------|--------------|
| `fc/insn` | FC instruction stream |
| `cluster/pe0/insn` | Cluster PE0 instructions |
| `cluster/pe0/lsu` | PE0 memory load/store events |
| `cluster/dma` | DMA transfer events |

### Tips
- Redirect trace to a file (`2>trace.txt`) — trace output goes to stderr
- Use `timeout 30 gvsoc ...` to limit trace duration
- Look at the **last few lines** of the trace to find where it's stuck
- Use `llvm-objdump -d <binary>` to map PC addresses to function names

## 8. Exercises

1. **Export and deploy SpeechNet inference** on Siracusa. Compare the ONNX node count with the training graph.

2. **Try `last_layer` training strategy** — only fine-tune the FC layer. Compare cycle count with full training.

3. **Increase training steps** — export with `--n-batches 16` (or `--n-steps 8 --n-accum 2`). Run on GVSoC and observe how loss evolves over more steps. Does it converge?

4. **Debug a hang**: Intentionally use `torch.flatten(x, 1)` in the model, export training ONNX, and observe what extra ops appear. Then fix it.

## 9. Reference

- [SilentWear paper](https://arxiv.org/abs/2603.02847)
- [Onnx4Deeploy repo](https://github.com/runwangdl/Onnx4Deeploy) — PR #2: SpeechNet exporter
- [TrainDeeploy repo](https://github.com/runwangdl/TrainDeeploy) — PR #31: SpeechNet training test
- [Deeploy TileConstraint docs](../AI_AGENT/Deeploy_Basics/Deeploy_TileConstraint.md)
- [Deeploy Kernel docs](../AI_AGENT/Deeploy_Basics/Deeploy_Kernel.md)